# E2 / E3 - Fine-tuned baselines on GPU

Everything else in this project runs locally on CPU. This notebook exists for the one part
that cannot: fine-tuning CNN backbones.

**What it does**
- E2 backbone selection on the *selection fold only* (fold 0), which never contributes to a reported number
- E3 baselines under the corrected patient-disjoint protocol, pooling out-of-fold predictions

**Why it matters for the paper.** The local protocol ladder used a frozen linear head, which
cannot memorise - so it cannot benefit from leakage, and it measures only a *lower bound* on
protocol inflation. A fine-tuned backbone can memorise. Re-running the ladder with these
predictions is what establishes the real inflation figure.

**Restartability.** Every epoch checkpoints to Drive and every finished model appends a row
to `model_rows.json`, which is skipped on re-run. A disconnect costs one epoch, never a run.

## 1. Environment and Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/ophthalmic-screening-research'

import os, sys, json, time, platform
import torch

print('GPU      :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('Python   :', platform.python_version())
print('PyTorch  :', torch.__version__)
print('CUDA     :', torch.version.cuda)
print('Drive    :', DRIVE_ROOT, '->', 'OK' if os.path.isdir(DRIVE_ROOT) else 'MISSING')

if not torch.cuda.is_available():
    print('WARNING: no GPU. Runtime > Change runtime type > T4 GPU, then rerun.')

## 2. What to upload to Drive first

Mirror these from the local repo into `MyDrive/ophthalmic-screening-research/`:

| Local | Drive |
|---|---|
| `data/manifests/odir_manifest_clean.csv` | `data/manifests/` |
| `data/splits/splits_*.csv` | `data/splits/` |
| the ODIR `Training Images` folder | `data/raw/odir_images/` |

The images are the large upload (~7 GB). Do it once; this notebook never writes to them.

In [ ]:
EXPERIMENT_ID  = 'E3_baselines_patient_disjoint_dedup'
SEED           = 42
IMAGE_SIZE     = 384
BATCH_SIZE     = 16
SPLIT_FILE     = 'splits_patient_disjoint_dedup.csv'
SELECTION_FOLD = 0

MANIFEST_DIR = DRIVE_ROOT + '/data/manifests'
SPLIT_DIR    = DRIVE_ROOT + '/data/splits'
IMAGE_DIR    = DRIVE_ROOT + '/data/raw/odir_images'
CKPT_DIR     = DRIVE_ROOT + '/checkpoints'
RESULT_DIR   = DRIVE_ROOT + '/results'
for d in (CKPT_DIR, RESULT_DIR):
    os.makedirs(d, exist_ok=True)

BACKBONES = ['resnet50', 'efficientnet_v2_s', 'convnext_tiny']
print('Experiment:', EXPERIMENT_ID, '| seed', SEED, '| backbones', BACKBONES)

## 3. Data

In [ ]:
import pandas as pd, numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

torch.manual_seed(SEED); np.random.seed(SEED)
LABELS = ['N', 'D', 'G', 'C']

splits = pd.read_csv(SPLIT_DIR + '/' + SPLIT_FILE)
splits['file'] = splits['original_filename']
print(len(splits), 'images |', splits['patient_id'].nunique(), 'patients')
print(splits.groupby('fold')['label'].value_counts().unstack().to_string())

# The one invariant worth re-asserting after any file transfer.
assert (splits.groupby('patient_id')['fold'].nunique() == 1).all(), 'a patient spans folds'
print('patient-disjoint: asserted OK')

In [ ]:
TRAIN_TF = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
EVAL_TF = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class FundusDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame, self.transform = frame.reset_index(drop=True), transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        img = Image.open(os.path.join(IMAGE_DIR, row['file'])).convert('RGB')
        return self.transform(img), LABELS.index(row['label'])

def loaders(train_df, val_df):
    tl = DataLoader(FundusDataset(train_df, TRAIN_TF), batch_size=BATCH_SIZE,
                    shuffle=True, num_workers=2, pin_memory=True)
    vl = DataLoader(FundusDataset(val_df, EVAL_TF), batch_size=BATCH_SIZE,
                    shuffle=False, num_workers=2, pin_memory=True)
    return tl, vl

## 4. Model and training loop

In [ ]:
import torch.nn as nn
from torchvision import models

def build(name):
    if name == 'resnet50':
        m = models.resnet50(weights='IMAGENET1K_V2')
        m.fc = nn.Linear(m.fc.in_features, 4)
    elif name == 'efficientnet_v2_s':
        m = models.efficientnet_v2_s(weights='IMAGENET1K_V1')
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 4)
    elif name == 'convnext_tiny':
        m = models.convnext_tiny(weights='IMAGENET1K_V1')
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, 4)
    else:
        raise ValueError(name)
    return m.cuda()

def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    total, preds, trues = 0.0, [], []
    for x, y in loader:
        x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
        with torch.set_grad_enabled(training), torch.autocast('cuda'):
            out = model(x)
            loss = criterion(out, y)
        if training:
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        total += loss.item() * len(y)
        preds.append(out.argmax(1).cpu())
        trues.append(y.cpu())
    return total / len(loader.dataset), torch.cat(preds), torch.cat(trues)

In [ ]:
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

def train_model(name, train_df, val_df, epochs=12, patience=4):
    tl, vl = loaders(train_df, val_df)
    model = build(name)
    weights = compute_class_weight('balanced', classes=np.arange(4),
                                   y=[LABELS.index(l) for l in train_df['label']])
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32).cuda())
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler()

    ckpt = CKPT_DIR + '/' + EXPERIMENT_ID + '_' + name + '_seed' + str(SEED) + '.pt'
    start, best, bad, history = 0, -1.0, 0, []
    if os.path.exists(ckpt):
        state = torch.load(ckpt, map_location='cuda')
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        start, best, history = state['epoch'] + 1, state['best'], state['history']
        print('  resumed from epoch', start)

    for epoch in range(start, epochs):
        t0 = time.time()
        run_epoch(model, tl, criterion, optimizer, scaler)
        _, vp, vt = run_epoch(model, vl, criterion)
        macro = float(f1_score(vt, vp, average='macro'))
        history.append({'epoch': epoch, 'val_macro_f1': macro})
        print('  epoch %2d  val macro-F1 %.4f  (%.0fs)' % (epoch, macro, time.time() - t0))
        scheduler.step()
        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'epoch': epoch, 'best': max(best, macro), 'history': history}, ckpt)
        if macro > best:
            best, bad = macro, 0
        else:
            bad += 1
            if bad >= patience:
                print('  early stop')
                break
    return model, best, history

## 5. E2 - backbone selection

Selection fold only. Whichever backbone wins here is locked before any test number exists,
so the selection fold never contributes to a reported result.

In [ ]:
ROWS_PATH = RESULT_DIR + '/model_rows.json'
rows = json.load(open(ROWS_PATH)) if os.path.exists(ROWS_PATH) else []
done = {(r['experiment'], r['backbone']) for r in rows}

sel_val   = splits[splits.fold == SELECTION_FOLD]
sel_train = splits[splits.fold != SELECTION_FOLD]

for name in BACKBONES:
    if ('E2', name) in done:
        print(name, 'already done, skipping')
        continue
    print('=== E2', name, '===')
    _, best, history = train_model(name, sel_train, sel_val)
    rows.append({'experiment': 'E2', 'backbone': name, 'seed': SEED,
                 'split_file': SPLIT_FILE, 'selection_fold': SELECTION_FOLD,
                 'best_val_macro_f1': best, 'history': history,
                 'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S')})
    json.dump(rows, open(ROWS_PATH, 'w'), indent=2)

e2 = [r for r in rows if r['experiment'] == 'E2']
if e2:
    table = pd.DataFrame(e2)[['backbone', 'best_val_macro_f1']]
    print(table.sort_values('best_val_macro_f1', ascending=False).to_string(index=False))

## 6. E3 - baselines under the corrected protocol

Cross-validation over the remaining folds with the chosen backbone. Out-of-fold predictions
are pooled and written back to Drive, so the local statistics scripts consume them with the
same code as every other experiment.

In [ ]:
CHOSEN = sorted(e2, key=lambda r: -r['best_val_macro_f1'])[0]['backbone'] if e2 else 'resnet50'
print('chosen backbone:', CHOSEN)

oof = []
for fold in sorted(splits.fold.unique()):
    if fold == SELECTION_FOLD:
        continue
    print('=== E3 fold', fold, '===')
    tr, te = splits[splits.fold != fold], splits[splits.fold == fold]
    model, best, _ = train_model(CHOSEN, tr, te)

    model.eval()
    probs = []
    loader = DataLoader(FundusDataset(te, EVAL_TF), batch_size=BATCH_SIZE, num_workers=2)
    with torch.no_grad(), torch.autocast('cuda'):
        for x, _ in loader:
            probs.append(torch.softmax(model(x.cuda()).float(), 1).cpu())
    probs = torch.cat(probs).numpy()

    block = te[['image_id', 'patient_id', 'label']].copy()
    block['y_true'] = block['label']
    block['y_pred'] = [LABELS[i] for i in probs.argmax(1)]
    for i, c in enumerate(LABELS):
        block['p_' + c] = probs[:, i]
    oof.append(block)

if oof:
    out = pd.concat(oof, ignore_index=True)
    out.to_csv(RESULT_DIR + '/E3_predictions_' + CHOSEN + '.csv', index=False)
    print('pooled out-of-fold predictions:', len(out))
    print('macro-F1:', round(float(f1_score(out.y_true, out.y_pred, average='macro')), 4))

## 7. Bring the results back

Download `results/E3_predictions_*.csv` and `results/model_rows.json` from Drive into the
local repo's `results/`, then run locally:

```bash
python scripts/protocol_ladder.py --predictions results/E3_predictions_<backbone>.csv
python scripts/make_figures.py
```

The fine-tuned ladder is the number that settles whether patient leakage inflates this
benchmark. The frozen-head result says only that *a low-capacity model gains nothing from
leakage* - a lower bound, not the answer.